# 🌌 NeoWatch — Aşama 3: Model Eğitimi, Optimizasyon ve Değerlendirme

**Plan Belgesi**: NASA Proje Planı 2 (Aşama 3)

### 🎯 Hedefler:
1. **Algoritma Seçimi & Baseline Modeller**: Lojistik Regresyon (baseline), Random Forest, LightGBM ve XGBoost algoritmalarını kurarak 5-Katlı Tabakalı Çapraz Doğrulama (Stratified 5-Fold CV) ile karşılaştırma.
2. **Hiperparametre Optimizasyonu**: `GridSearchCV` kullanarak XGBoost parametrelerini (`max_depth`, `learning_rate`, `n_estimators`, `scale_pos_weight`) en iyi hale getirme.
3. **İş Problemine Uygun Metrik Seçimi**: Gezegensel savunmada tehlikeli cismi kaçırmanın (False Negative) bedeli yıkıcı olduğundan, **Recall (Duyarlılık)** ve **ROC-AUC** skorlarını maksimize etme.
4. **Karmaşıklık Matrisi (Confusion Matrix)**: Modelin Hata Dağılımını analiz etme.
5. **Checkpoint 3 (Zeka Katmanı)**: Yüksek Recall başarısına ulaşan modeli `models/asteroid_xgb_model.pkl` adıyla kaydetme.

In [ ]:
import sys
import os
from pathlib import Path

project_root = Path(os.path.abspath('')).parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from src.config import (
    RAW_DATA_PATH,
    LEGACY_RAW_DATA_PATH,
    PROCESSED_DATA_PATH,
    MODEL_PATH,
    LEGACY_MODEL_PATH,
    FEATURE_COLUMNS,
    TARGET_COLUMN,
)
from src.data_processor import DataProcessor
from src.model_trainer import AsteroidModelTrainer

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Aşama 3 Çalışma Ortamı Hazırlandı!")

## 1. Veri Setinin Hazırlanması (Önişleme & SMOTE)

In [ ]:
raw_path = RAW_DATA_PATH if RAW_DATA_PATH.exists() else LEGACY_RAW_DATA_PATH
processor = DataProcessor(scaler_type='standard')

X_train, y_train, X_test, y_test, df_clean = processor.prepare_datasets(
    raw_csv_path=raw_path,
    test_size=0.2,
    random_state=42,
    apply_smote=True,
)

print(f"Eğitim Seti (SMOTE dengeli): {X_train.shape}")
print(f"Test Seti (Holdout)         : {X_test.shape}")

## 2. Baseline Model Kıyaslaması (5-Fold Stratified Cross Validation)

In [ ]:
trainer = AsteroidModelTrainer(random_state=42)
df_benchmarks = trainer.run_benchmarks(X_train, y_train, cv_splits=5)
print("--- 5-Fold Cross Validation Karşılaştırma Tablosu ---")
df_benchmarks

## 3. XGBoost Hiperparametre Optimizasyonu (GridSearchCV - Recall Odaklı)

In [ ]:
print("GridSearchCV ile XGBoost hiperparametre optimizasyonu başlatılıyor...")
best_xgb = trainer.tune_xgboost(X_train, y_train)
print("Optimizasyon tamamlandı!")

## 4. Test Seti Değerlendirmesi & Karmaşıklık Matrisi (Confusion Matrix)

In [ ]:
eval_results = trainer.evaluate_model(best_xgb, X_test, y_test)

# Karmaşıklık Matrisi Görselleştirme
plt.figure(figsize=(6, 5))
sns.heatmap(
    eval_results['confusion_matrix'],
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Tahmin: Güvenli (0)', 'Tahmin: Tehlikeli (1)'],
    yticklabels=['Gerçek: Güvenli (0)', 'Gerçek: Tehlikeli (1)'],
)
plt.title(f"Test Seti Karmaşıklık Matrisi (Recall: %{eval_results['recall']*100:.2f})")
plt.ylabel('Gerçek Sınıf')
plt.xlabel('Tahmin Edilen Sınıf')
plt.tight_layout()
plt.show()

print("--- Final Test Seti Skorları ---")
print(f"Recall (Duyarlılık) : %{eval_results['recall']*100:.2f} (Planetary Defense Hedefi: >= %90)")
print(f"ROC-AUC Skoru       : {eval_results['roc_auc']:.4f}")
print(f"F1 Skoru            : {eval_results['f1']:.4f}")
print(f"Precision           : %{eval_results['precision']*100:.2f}")

## 5. CHECKPOINT 3: Zeka Katmanı ve Model Kaydı (`models/asteroid_xgb_model.pkl`)

In [ ]:
# Modeli hem Plan 2 hem de legacy dosya yoluna kaydetme
trainer.save_model(MODEL_PATH)
print(f"\n🎯 CHECKPOINT 3 TAMAMLANDI: Eğitilmiş XGBoost modeli kaydedildi -> {MODEL_PATH}")